In [1]:
import simpy
import numpy as np
import csv
import json
import os
import multiprocessing
import pandapower.networks as pn

from tqdm import tqdm
from datetime import datetime

from datacenter import DataCenter
from grid_regions import GridRegion
from util import environment_updater, load_job_trace, datacenter_state_logger, init_users

from simulation_interface import GridSimulation
from sim_config import GridConfig, DataCenterConfig

from config import (
    UNIT_PRICES, EGRESS_RATE, BANDWIDTH, HARDWARE_MODELS, 
    RTT_MATRIX, NETWORK_ENERGY_PER_GB, WORKLOAD_TRACE_LENGTH_DAYS, SIMULATION_START_TIME, AWARE_QUEUE, 
    ALLOCATION_FAILED_RETRY, DECISION_METHOD, ELECTRICITY_PRICES
)

In [ ]:
SEED = 41
SHIFTING_MODE = 'none'
OVERPROVISIONING_FACTOR = 4
WORKLOAD_TRACE_LENGTH_DAYS = 7
AWARE_QUEUE = False
ALLOCATION_FAILED_RETRY = True
CARBON_SCALING_FACTOR = 1.0

# Input files
user_datacenter_config_path = f"/mnt/grid-cloud-migration-estimates_copy/simulator/src/case_1/input/simulation_config_811_seed_{SEED}.json"

# Output files
job_result_path = f"output_test/job_result_{SHIFTING_MODE}.csv"
datacenter_result_path = f"output_test/datacenter_result_{SHIFTING_MODE}.csv"

In [ ]:
# ==========================================
# Main Simulation
# ==========================================

if __name__ == "__main__":    
    env = simpy.Environment()
    
    env.shifting_mode = SHIFTING_MODE
    env.egress_rate = EGRESS_RATE
    env.bandwidth = BANDWIDTH
    env.hardware_model = HARDWARE_MODELS
    env.simulation_start_time = datetime.strptime('2020-08-01 00:00:00', '%Y-%m-%d %H:%M:%S')
    env.rtt_matrix = RTT_MATRIX
    env.network_energy_per_gb = NETWORK_ENERGY_PER_GB
    env.workload_trace_length_days = WORKLOAD_TRACE_LENGTH_DAYS
    env.aware_queue = AWARE_QUEUE
    env.allocation_failed_retry = ALLOCATION_FAILED_RETRY
    env.decision_method = DECISION_METHOD
    env.carbon_scaling_factor = CARBON_SCALING_FACTOR
    
    env.datacenters = []
    with open(user_datacenter_config_path, 'r') as f:
        config = json.load(f)
    datacenter_capacity_config = config['regions']
     # First pass: build grid configs only
    grid_configs: list[GridConfig] = []
    for dc_name, _ in datacenter_capacity_config.items():
        grid_config = GridConfig(
            pp_grid = pn.case118(), # type: ignore
            region  = GridRegion(dc_name)
        )
        grid_config.add_data_center(DataCenterConfig(
            load_share                  = 0.5,
            onsite_overprovision_factor = 1.0
        ))
        grid_configs.append(grid_config)

    grid_sim_interface: GridSimulation = GridSimulation(
        grid_configs=grid_configs,
        shifting_threshold=0.0,
        enable_weather_variation=True
    )
    env.grid_simulator_interface: GridSimulation = grid_sim_interface # type: ignore

    # Second pass: create DataCenter instances with the real interface
    env.datacenters = []
    capacity_factor = 1.1
    
    for dc_name, dc_config in datacenter_capacity_config.items():
        gpu_cap = int(dc_config.get('gpu_slots_capacity', 100) * capacity_factor)
        cpu_cap = int(dc_config.get('cpu_capacity', 100) * capacity_factor)
        mem_cap = int(dc_config.get('mem_capacity', 100) * capacity_factor)

        grid_trace_path = f"/mnt/grid-cloud-migration-estimates_copy/data/grid/processed/{dc_name.replace('_', '-')}.csv"

        dc = DataCenter(
            env,
            dc_name,
            capacity=[gpu_cap, cpu_cap, mem_cap],
            unit_resource_prices=UNIT_PRICES[dc_name],
            pue=1.2,
            grid_trace_path=grid_trace_path,
            simulation_interface=grid_sim_interface,
            electricity_prices=ELECTRICITY_PRICES[dc_name]
        )
        env.datacenters.append(dc)
        
    # Initialize users
    env.users_dict = {}
    
    # Setup users and jobs
    init_users(env, user_datacenter_config_path)
    load_job_trace(env, user_datacenter_config_path)
    env.process(environment_updater(env))

    print(f"Total jobs loaded: {env.num_total_job}")
    
    # Start the coroutine to update environment conditions (carbon intensity for now, may add more factors later)
    print("Grid trace loaded and environment updater process started.")
    
    # Set up a simulation end condition based on total number of jobs completed
    env.simulation_done = env.event()
    # Add a progress bar to track job completion
    env.pbar = tqdm(total=env.num_total_job, desc="Jobs Completed")
    
    # Stream writing file handles
    env.job_scheduling_result_buffer = []
    env.datacenter_state_buffer = []
    env.buffer_flush_threshold = 1000  # Flush to disk every 1000 records
    env.datacenter_state_log_interval = 60  # Log datacenter state every 60 seconds
    
    f_job_scheduling = open(job_result_path, "w", newline="")
    env.results_writer = csv.DictWriter(f_job_scheduling, fieldnames=[
        "user_id", "job_id", 
        "home_dc", "target_dc", 
        "submit_time", 
        "cost_threshold", "delay_threshold", "temporal_slack",
        "num_retry", "temporal_delay", "failed_retry_delay", "shift_delay", "queue_delay", "duration", 
        "shift_cost", "compute_cost",
        "expected_shift_carbon", "expected_compute_carbon",
        "shift_carbon", "compute_carbon", 
        "expected_score", "score",
    ])
    env.results_writer.writeheader()
    
    f_datacenter_state = open(datacenter_result_path, "w", newline="")
    env.timeseries_writer = csv.DictWriter(f_datacenter_state, fieldnames=[
        "time", "time_human",
        "dc_name", "carbon_intensity",
        "cpu_total", "cpu_used", 
        "gpu_total", "gpu_used", 
        "mem_total", "mem_used", 
        "power",
        "peak_power",
        "cumulative_energy_consumed",
        "cumulative_energy_cost",
        "queue_length",
    ])
    env.timeseries_writer.writeheader()
    
    try:
        env.process(datacenter_state_logger(env))
        
        # Run the simulation until the end event
        env.run(until=env.simulation_done)
        
        # Flush remaining results in the buffer to disk
        if env.job_scheduling_result_buffer:
            env.results_writer.writerows(env.job_scheduling_result_buffer)
        if env.datacenter_state_buffer:
            env.timeseries_writer.writerows(env.datacenter_state_buffer)
    finally:
        f_job_scheduling.close()
        f_datacenter_state.close()
        env.pbar.close()
        print(f"Simulation completed at time: {env.now}")

Total jobs loaded: 283773
Grid trace loaded and environment updater process started.


Jobs Completed:  71%|███████▏  | 202784/283773 [08:07<02:42, 499.29it/s] 